# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1 -> Ответ: [1, 2, 3] [0, 1, 1] в массивах идет работа с числами внутри него, поэтому каждый элемент делится умножается и тд 
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2 -> 1 1 1 1
#      1 1 1 1 
#      1 1 1 1  создали 3 строки 4 столбца массив из 1 
# Ответ:
#      4
#      4 .sum(axis=1) выводит сумму по столбцам 
#      4 неправильно отобразила 
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3 Ответ:  
# 0 1 2 + 0 0 0     0 1 2
# 0 1 2 + 1 1 1 ->  1 2 3
# 0 1 2 + 2 2 2     2 3 4
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4 Ответ: 1 4 9 -> 1 + 2 + 3 = 6
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5 Ответ: 1 2 3 4 **2 -> 1 4 9 16 -> 1+3+9+16=30/4=7,5
print((np.arange(1, 5) ** 2).mean())
```

In [7]:
import numpy as np
a = np.array([2, 4, 6]); print(a / 2, a // 4)
m = np.ones((3, 4)); print(m.sum(axis=1))
print(np.arange(3) + np.arange(3).reshape(3, 1))
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
print((np.arange(1, 5) ** 2).mean())


[1. 2. 3.] [0 1 1]
[4. 4. 4.]
[[0 1 2]
 [1 2 3]
 [2 3 4]]
6.0
7.5


In [8]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [9]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [10]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [16]:
#для проверки 
print(clean)
print(month)

[ True  True  True ...  True False  True]
['2026-09' '2026-09' '2026-09' ... '2026-10' '2026-10' '2026-10']


In [29]:
# Ответ
print(f"В clean осталось {clean.sum()} строк")
september = amount[clean & (month == "2026-09")].sum()
october = amount[clean & (month == "2026-10")].sum()
print(f"Выручка за сентябрь: {september} \nВыручка за октябрь: {october}  \nВыручка ниже на {round(100- (october/september*100),2)}% в октябре, сравнивая с прошлым месяцем")

В clean осталось 4739 строк
Выручка за сентябрь: 35243410.0 
Выручка за октябрь: 34419520.0  
Выручка ниже на 2.34% в октябре, сравнивая с прошлым месяцем


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [81]:
# 2
cats = np.unique(cat)
cat_month = []
for c in cats:
    amount_sep = (cat == c) & clean & (month == "2026-09")
    amount_oct = (cat == c) & clean & (month == "2026-10")
    cat_month.append([amount[amount_sep].sum(), amount[amount_oct].sum()])

cat_month_table = np.array(cat_month)
total = cat_month_table.sum(axis=1)
change = 100-cat_month_table[:, 1:]/cat_month_table[:, :1]*100
print(f"Какая категория приносит больше всего выручки за два месяца (`axis`)? \n{cats[total.argmax()]}: {total.max()} тенге")
print(f"Какая упала сильнее всех в %? \n{cats[change.argmin()]}: {round(change.min(),2)}%")

Какая категория приносит больше всего выручки за два месяца (`axis`)? 
Электроника: 34370430.0 тенге
Какая упала сильнее всех в %? 
Книги: -7.1%


In [104]:
#для себя
print(cat_month_table)
cat_month_table[:, 1:]

cat_month_table[:, :1]
print(change)

[[ 4997730.  5125790.]
 [ 1065300.  1140910.]
 [ 2610950.  2678990.]
 [ 9167100.  8505730.]
 [17402330. 16968100.]]
[[-2.56236331]
 [-7.09753121]
 [-2.60594803]
 [ 7.2146044 ]
 [ 2.49524058]]


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [ ]:
# 3
cats = np.unique(cat)
cat_month_count = []
for c in cats:
    amount_sep = (cat == c) & clean & (month == "2026-09")
    amount_oct = (cat == c) & clean & (month == "2026-10")
    cat_month_count.append([amount_sep.sum(), amount_oct.sum()])
cat_month_count_arr = np.array(cat_month_count)
average = cat_month_table/cat_month_count_arr
average_oct = average[:, :1]
print(f'У какой категории самый высокий cредний чек в октярбе? {cats[average_oct.argmax()]}, {average_oct.max()} тенге')

У какой категории он самый высокий в октярбе? Электроника, 32772.74952919021 тенге


In [113]:
#для себя
print(cat_month_count_arr)
print(average_oct)


[[476 476]
 [265 299]
 [422 435]
 [678 652]
 [531 505]]
[[10499.43277311]
 [ 4020.        ]
 [ 6187.08530806]
 [13520.79646018]
 [32772.74952919]]


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [136]:
# 4
cities = np.unique(city)
months = np.unique(month)


table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months]
                  for c in cities])
share = table / table.sum(axis=0)
share.sum(axis=0)
shym = share[cities == "Шымкент"]*100 
print(f"Доля Шымкента в сентябре: {round(shym[0][0], 2)}% В октябре: {round(shym[0][1],2)}%")

Доля Шымкента в сентябре: 12.93% В октябре: 11.75%


In [ ]:
# для себя
print(table)
print( table[:, 1:])


[[ 2576960.  2540420.]
 [14501070. 14643020.]
 [ 9034220.  8850530.]
 [  923990.   969860.]
 [ 3648520.  3371680.]
 [ 4558650.  4044010.]]
[[ 2540420.]
 [14643020.]
 [ 8850530.]
 [  969860.]
 [ 3371680.]
 [ 4044010.]]


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [161]:
# 5
change_cities = table[:, 1] - table[:, 0] 
share_cities = change_cities / change_cities.sum(axis=0)
print(f" Шымкент дает {round(share_cities[5]*100,2)}% падения") 

 Шымкент дает 62.46% падения


In [ ]:
#для себя
print(change_cities)
print(share_cities)
share.sum(axis=0)

[[ -36540.]
 [ 141950.]
 [-183690.]
 [  45870.]
 [-276840.]
 [-514640.]]
[[ 0.04435058]
 [-0.17229242]
 [ 0.22295452]
 [-0.05567491]
 [ 0.33601573]
 [ 0.62464649]]


array([1., 1.])

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [ ]:
# 6 разные формы keepdims оставтт форму 6 1
print(table.shape)
print(table.sum(axis=1).shape)

share_in_city = table / table.sum(axis=1, keepdims=True)
print(share_in_city)


(6, 2)
(6,)
[1. 1. 1. 1. 1. 1.]
[[0.50357019 0.49642981]
 [0.49756469 0.50243531]
 [0.50513538 0.49486462]
 [0.48788975 0.51211025]
 [0.51971739 0.48028261]
 [0.52991168 0.47008832]]


## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8